# core

> Fill in a module description here

In [ ]:
# | default_exp extraction

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [ ]:
# # | export
# from enum import Enum
# from typing import List, Union, Optional
# from pydantic import BaseModel, Field
# from instructor import patch, Mode
# from litellm import LiteLLM, acompletion
# import litellm
# from dotenv import load_dotenv

# litellm.drop_params = True

# load_dotenv()

True

In [ ]:
# # | export
# # custom ALiteLLM class to patch acompletion support, which isn't available in vanilla litellm-instructor integration. https://github.com/jxnl/instructor/blob/b06eb4faa32e7b5e0f3628a611490efd042868fa/docs/blog/posts/learn-async.md#L4
# class ALiteLLM(LiteLLM):
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.chat.completions.create = self._patched_acompletion

#     async def _patched_acompletion(self, *args, **kwargs):
#         return await acompletion(
#             *args,
#             **kwargs
#         )

In [ ]:
# class test(BaseModel):
#     name: str

# client = patch(ALiteLLM(), mode=Mode.JSON)

# await client.chat.completions.create(
#     model="gpt-4-0125-preview",
#     response_format={"type": "json_object"},
#     response_model=test,
#     max_retries=1,
#     max_tokens=4096,
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": f"What is the name of the person speaking? Reply with json: {test.model_json_schema()}."},
#     ],
# )

test(name='unknown')

In [ ]:
# test.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'}},
 'required': ['name'],
 'title': 'test',
 'type': 'object'}

In [ ]:
# # | export


# class ModelType(Enum):
#     CLAUDE_3_HAIKU = "claude-3-haiku-20240307"
#     CLAUDE_3_SONNET = "claude-3-sonnet-20240229"
#     GPT_4_0125_PREVIEW = "gpt-4-0125-preview"
#     GPT_3_5_TURBO = "gpt-3.5-turbo"
#     GPT_4_TURBO = "gpt-4-turbo-2024-04-09"
#     CLAUDE_3_OPUS = "claude-3-opus-20240229"
#     GPT_4o = "gpt-4o"


# class QuestionSchema(BaseModel):
#     id: str
#     text: str
#     output_type: str
#     categories: Optional[List[str]] = None


# class Questions(BaseModel):
#     questions: List[QuestionSchema]


# class AnswerSchema(BaseModel):
#     question_id: str
#     text: str = Field(..., title="The answer to the question")

#     def __hash__(self):
#         return hash(self.question_id)


# class AnswerCategorySchema(BaseModel):
#     question_id: str
#     categories: Union[str, List[str]] = Field(
#         ...,
#         title="The answer to the question. Must be a valid category or list of category from the originally provided text.",
#     )


# class Answers(BaseModel):
#     answers: List[Union[AnswerSchema, AnswerCategorySchema]]

#     def __hash__(self):
#         return hash(self.answers[0].question_id)


# class AIModelClient:
#     """
#     A client for interacting with various language models to create schemas and extract information.

#     Args:
#         model_type (ModelType): The type of language model to use.
#     """

#     def __init__(
#         self,
#         model_type: ModelType = ModelType.GPT_4o,
#         max_retries: int = 10,
#         max_tokens: int = 4096,
#     ):
#         self.client = patch(ALiteLLM(), mode=Mode.JSON)  # type: ignore
#         self.model_type = model_type
#         self.max_tokens = max_tokens
#         self.max_retries = max_retries
#         self.start_with = "{"

#     async def create_schema(
#         self, questions: str, model_type_override: Optional[ModelType] = None
#     ) -> Questions:
#         """
#         Create a schema from the given questions.

#         Args:
#             questions (str): The questions to be structured.

#         Returns:
#             Questions: The structured questions.
#         """
#         EXTRACTION_PROMPT_B = f"Extract a set of structured questions from the user-provided unstructured text using ONLY JSON with NO preamble so it can be validated immediately, with no spacing or newlines, following the following JSON schema format: {Questions.model_json_schema()}. i.e., start your response with {self.start_with}. If the information requested in the question is absent, leave the answer completely blank or write 'N/a'. You should respond in a message, not a tool or function call, if that's an option for you. Output type can be 'string', or 'category'. If output type is 'category', then the categories key should be a list of relevant categories provided by the user."

#         response: Questions = await self.client.chat.completions.create(  # type: ignore
#             model=model_type_override.value
#             if model_type_override
#             else self.model_type.value,
#             response_model=Questions,
#             max_retries=self.max_retries,
#             max_tokens=self.max_tokens,
#             messages=[
#                 {"role": "system", "content": EXTRACTION_PROMPT_B},
#                 {
#                     "role": "user",
#                     "content": questions,
#                 },
#             ],
#         )
#         if isinstance(response, Questions):
#             return response
#         else:
#             raise ValueError("Response is not of type Questions")

#     async def extract_information(
#         self,
#         text: str,
#         questions: Questions,
#         model_type_override: Optional[ModelType] = ModelType.GPT_4o,
#     ) -> Answers:
#         """
#         Extract information from the given text using the provided schema.

#         Args:
#             text (str): The text to extract information from.
#             schema (Questions): The schema to use for extraction.

#         Returns:
#             Answers: The extracted information.
#         """
#         EXTRACTION_PROMPT_C = f"Answer the following questions based on the user-provided text using ONLY JSON with NO preamble so it can be validated immediately, with no spacing or newlines, following the following JSON schema format: {Answers.model_json_schema()}, i.e., start your response with {self.start_with}. You should respond in a message, not a tool or function call, if that's an option for you. Be concise, and answer only the question or questions provided: {questions.model_dump_json()}. Answer these questions in the most intelligent way possible, as this is high-stakes for the user to get it right. Provide direct supporting quotes where possible."
#         EXTRACTION_PROMPT_A = f"You are a product analyst assistant. Please look at the transcripts provided and give careful responses for each field, explaining the answer and your rationale to the question in detail while also providing 1-3 direct quotes from the transcript when relevant, and edited lightly if needed for brevity and clarity.\n\nBe as comprehensive as possible, providing all relevant information.\n\nHere's the scheme you must respond in, it will be validated with pydantic and so you MUST respond only with the JSON.\n\nUse Null if the answer is not present in the transcript, not every question is asked each time.\n\nUse context clues to understand who the researcher is in the conversation and which speaker is being interviewed.{questions.model_dump_json()} Finally, If the answer is not provided in the transcript provided, say 'N/a', unless the question says that it's required by the user — in that case, provide your best analysis based on the provided info."

#         response: Answers = await self.client.chat.completions.create(  # type: ignore
#             model=model_type_override.value
#             if model_type_override
#             else self.model_type.value,
#             response_model=Answers,
#             max_retries=self.max_retries,
#             max_tokens=self.max_tokens,
#             messages=[
#                 {"role": "system", "content": EXTRACTION_PROMPT_A},
#                 {"role": "user", "content": text},
#             ],
#             response_format={"type": "json_object"},
#         )
#         if isinstance(response, Answers):
#             return response
#         else:
#             raise ValueError("Response is not of type Answers")

In [ ]:
# # | export
# import pandas as pd
# from io import BytesIO


# class QADataManager:
#     def __init__(self):
#         self.df = pd.DataFrame()

#     def add_answers(
#         self, questions: Questions, answers: Answers, document_name: str, user_id: str
#     ):
#         data = {"document_name": document_name, "user_id": user_id}
#         for question in questions.questions:
#             answer = next(
#                 (a for a in answers.answers if a.question_id == question.id), None
#             )
#             if answer:
#                 if isinstance(answer, AnswerSchema):
#                     data[question.text] = answer.text
#                 else:
#                     # Ensure categories is a list before joining
#                     categories = (
#                         answer.categories
#                         if isinstance(answer.categories, list)
#                         else [answer.categories]
#                     )
#                     data[question.text] = ", ".join(categories)

#         new_df = pd.DataFrame([data])
#         self.df = pd.concat([self.df, new_df], ignore_index=True)

#     def to_csv(self, file_path: str):
#         self.df.to_csv(file_path, index=False)

#     def to_csv_bytes(self) -> bytes:
#         csv_buffer = BytesIO()
#         self.df.to_csv(csv_buffer, index=False)
#         csv_bytes = csv_buffer.getvalue()
#         return csv_bytes
    
#     def merge_rows_by_user(self):
#         """Merge rows by user_id, concatenating the data for each user. Document row is also concatenated."""
#         if 'user_id' not in self.df.columns:
#             raise ValueError("DataFrame does not contain a 'user_id' column.")
#         def concat_documents(docs: "pd.Series[str]"):
#             # Filter out 'N/a', 'na', 'nan', 'null', and actual NaN values based on lowered case without altering original case
#             filtered_docs = docs[docs.str.lower().str.strip().replace(['n/a', 'na', 'nan', 'null', ''], pd.NA).notna()]
#             if len(filtered_docs) == 1:
#                 return filtered_docs.iloc[0]
#             return ',\n'.join(f"{i+1}. {doc}" for i, doc in enumerate(filtered_docs))

#         aggregation_functions = {col: concat_documents for col in self.df.columns if col != 'user_id'}
#         self.df = self.df.groupby('user_id').agg(aggregation_functions).reset_index()


In [ ]:
# testdf = pd.DataFrame({
#         'document_name': ['Doc1', 'Doc2', 'doc3', 'doc4'],
#         'user_id': ['user1', 'usEr1', 'user2', 'user2'],
#         'data': ['data1', 'dAta2', 'data3', 'nuLL'],
#         'data2': ['Data1', 'N/a', 'data3', 'data4'],
#     })

# test_df = QADataManager()
# test_df.df = testdf
# test_df.merge_rows_by_user()
# print(test_df.df)
# test_df.to_csv("test_df_merge.csv")


  user_id      document_name   data                data2
0   usEr1               Doc2  dAta2                     
1   user1               Doc1  data1                Data1
2   user2  1. doc3,\n2. doc4  data3  1. data3,\n2. data4


In [ ]:
# import os
# import time
# import asyncio

# EXAMPLE_TEXT = """Speaker A: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US. Skylines from Maine to Maryland to Minnesota are gray and smoggy. And in some places, the air quality warnings include the warning to stay inside. We wanted to better understand what's happening here and why, so we called Peter DiCarlo, an associate professor in the department of Environmental Health and Engineering at Johns Hopkins University. Good morning. Professor Good morning. So what is it about the conditions right now that have caused this round of wildfires to affect so many people so far away?
# Speaker B: Well, there's a couple of things. The season has been pretty dry already, and then the fact that we're getting hit in the US is because there's a couple of weather systems that are essentially channeling the smoke from those Canadian wildfires through Pennsylvania into the mid Atlantic and the northeast and kind of just dropping the smoke there.
# Speaker A: So what is it in this haze that makes it harmful? And I'm assuming it is.
# Speaker B: Is it is. The levels outside right now in Baltimore are considered unhealthy. And most of that is due to what's called particulate matter, which are tiny particles, microscopic, smaller than the width of your hair, that can get into your lungs and impact your respiratory system, your cardiovascular system, and even your neurological, your brain.
# Speaker A: What makes this particularly harmful? Is it the volume of particulate? Is it something in particular? What is it exactly? Can you just drill down on that a little bit more? Yeah.
# Speaker B: So the concentration of particulate matter I was looking at, some of the monitors that we have was reaching levels of what are, in science speak, 150 micrograms per meter cubed, which is more than ten times what the annual average should be and about four times higher than what you're supposed to have on a 24 hours average. And so the concentrations of these particles in the air are just much, much higher than we typically see. And exposure to those high levels can lead to a host of health problems.
# Speaker A: And who is most vulnerable? I noticed that in New York City, for example, they're canceling outdoor activities. And so here it is in the early days of summer, and they have to keep all the kids inside. So who tends to be vulnerable in a situation like this?
# Speaker B: It's the youngest. So children, obviously, whose bodies are still developing, the elderly who know their bodies are more in decline and they're more susceptible to the health impacts of breathing, the poor air quality. And then people who have preexisting health conditions, people with respiratory conditions or heart conditions, can be triggered by high levels of air pollution.
# Speaker A: Could this get worse?
# Speaker B: That's a good, in some areas it's much worse than others, and it just depends on kind of where the smoke is concentrated. I think New York has some of the higher concentrations right now, but that's going to change as that air moves away from the New York area. But over the course of the next few days, we will see different areas being hit at different times with the highest concentrations. I was going to ask you more fires start burning. I don't expect the concentrations to go up too much higher.
# Speaker A: I was going to ask you, and you started to answer this, but how much longer could this last? Or, forgive me if I'm asking you to speculate, but what do you think?
# Speaker B: Well, I think the fires are going to burn for a little bit longer, but the key for us in the US is the weather system changing. And so right now it's kind of the weather systems that are pulling that air into our mid atlantic and northeast region. As those weather systems change and shift, we'll see that smoke going elsewhere and not impact us in this region as much. And so I think that's going to be the defining factor. And I think the next couple of days we're going to see a shift in that weather pattern and start to push the smoke away from where we are.
# Speaker A: And finally, with the impacts of climate change, we are seeing more wildfires. Will we be seeing more of these kinds of wide ranging air quality consequences or circumstances?
# Speaker B: I mean, that is one of the predictions for climate change. Looking into the future, the fire season is starting earlier and lasting longer, and we're seeing more frequent fires. So, yeah, this is probably something that we'll be seeing more frequently. This tends to be much more of an issue in the western Us. So the eastern us getting hit right now is a little bit new. But, yeah, I think with climate change moving forward, this is something that is going to happen more frequently.
# Speaker A: That's Peter DiCarlo, associate professor in the department of Environmental Health and engineering at Johns Hopkins University. Thanks so much for joining us and sharing this expertise with us.
# Speaker B: Thank you for having me."""

# questions = """1. What is the name of speaker A, if mentioned?
# 2. What is the name of speaker B, if mentioned?
# 3. What are they talking about?
# 4. Pick from the following list of tags: smoke, air quality, health, climate change, and pick as many as are relevant that apply to the conversation."""


# async def test_model_client_create():
#     ai_client = AIModelClient()

#     # Test create_schema
#     schema = await ai_client.create_schema(questions)

#     assert isinstance(schema, Questions)
#     assert len(schema.questions) == 4
#     assert schema.questions[0].id == "1"
#     assert "What is the name of speaker A" in schema.questions[0].text
#     assert schema.questions[0].output_type == "string"
#     assert len(schema.questions[3].categories) == 4
#     return schema


# async def test_model_client_extract(schema: Questions):
#     """Going to be a bit flaky if using haiku"""
#     ai_client = AIModelClient(ModelType.CLAUDE_3_HAIKU)
#     # Test extract_information
#     print(schema)
#     answers = await ai_client.extract_information(EXAMPLE_TEXT, schema)

#     assert isinstance(answers, Answers)
#     print(answers)
#     assert len(answers.answers) == 4
#     assert answers.answers[0].question_id == "1"
#     assert len(answers.answers[0].text) > 0
#     assert len(answers.answers[3].categories) > 0
#     return answers


# async def test_qa_dataframe_manager(questions: Questions, answers: Answers):
#     qa_manager = QADataManager()

#     qa_manager.add_answers(questions, answers, "example_doc.txt", "user123")

#     assert len(qa_manager.df) == 1
#     assert qa_manager.df["document_name"].tolist() == ["example_doc.txt"]
#     assert qa_manager.df["user_id"].tolist() == ["user123"]
#     for question in schema.questions:
#         assert question.text in qa_manager.df.columns

#     qa_manager.to_csv("test_qa_data.csv")
#     assert os.path.exists("test_qa_data.csv")
#     time.sleep(1)
#     loaded_df = pd.read_csv("test_qa_data.csv")  # type: ignore
#     assert loaded_df.equals(qa_manager.df)  # type: ignore


# schema = await test_model_client_create()
# answers = await test_model_client_extract(schema)
# await test_qa_dataframe_manager(schema, answers)



questions=[QuestionSchema(id='1', text='What is the name of speaker A, if mentioned?', output_type='string', categories=None), QuestionSchema(id='2', text='What is the name of speaker B, if mentioned?', output_type='string', categories=None), QuestionSchema(id='3', text='What are they talking about?', output_type='string', categories=None), QuestionSchema(id='4', text='Pick from the following list of tags: smoke, air quality, health, climate change, and pick as many as are relevant that apply to the conversation.', output_type='category', categories=['smoke', 'air quality', 'health', 'climate change'])]
answers=[AnswerSchema(question_id='1', text='N/a'), AnswerSchema(question_id='2', text='Peter DiCarlo'), AnswerSchema(question_id='3', text='They are discussing the smoke from Canadian wildfires affecting air quality across the US, the factors contributing to this situation, the health impacts of air pollution exacerbated by such smoke, the populations most vulnerable to these condition

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore